In [1]:
%load_ext autoreload
%autoreload 2

In [30]:
import os
import sys
import json
from pathlib import Path

os.chdir("/home/kaariaa3/mscthesis/")
sys.path.append("./src/")  # Add module directory to path

import pandas as pd
import numpy as np

from utils.plots import calculate_metrics, calculate_accuracy
from utils.tools import get_config, collect_jobs, prettify_table, format_table, aggregate_results
from utils.constants import PRED_COLS

In [3]:
cols = [
    "model",
    "use_instructions",
    "type_of_demonstrations",
    "number_of_demonstrations",
    "total_accuracy",
    "theme_accuracy",
    "theme_precision",
    "theme_recall",
    "theme_f1",
    "topic_accuracy",
    "topic_precision",
    "topic_recall",
    "topic_f1",
    "concept_accuracy",
    "concept_precision", 
    "concept_recall",
    "concept_f1",
]

def score_table(jobs, cols, measure_hallucination_detection=True, mark_gen_errors_as_hallucinations=False):

    print(f"Processing {len(jobs)} jobs.")    
    table = {col_name: [] for col_name in cols}
    num_parse_errors = 0
    num_gen_errors = 0
    num_rows = 0
    for job in jobs:
        df = pd.read_csv(job["result_csv"], sep=";")

        num_rows += len(df)

        # LLM failed to produce proper json
        mask_parse_errors = ~(df[PRED_COLS] != '"PARSE ERROR"').all(axis=1)
        num_parse_errors += list(mask_parse_errors).count(True)

        # LLM did not follow instructions, also includes parse errors
        mask_gen_errors = ~df[PRED_COLS].isin(["\"yes\"", "\"no\""]).all(axis=1)
        num_gen_errors += list(mask_gen_errors).count(True) - list(mask_parse_errors).count(True)

        if mark_gen_errors_as_hallucinations:
            # Set labels
            df.loc[mask_gen_errors, ['themeCorrect', 'topicCorrect', 'usesAdditionalConcepts']] = ['"no"', '"no"', '"yes"']
        else: 
            # Remove erroneous rows
            df = df[~mask_gen_errors]

        config = get_config(job["config_json"])

        # Append run configs
        table["model"].append(job["model"])
        table["use_instructions"].append(config["use_instructions"])
        table["type_of_demonstrations"].append(config["type_of_demonstrations"])
        table["number_of_demonstrations"].append(config["number_of_demonstrations"])

        # Append accuracy scores
        for name, score in calculate_accuracy(df).items():
            table[name].append(score)

        # Append precision, recall, f1 per label
        metrics = calculate_metrics(df)
        for name, score in calculate_metrics(df, measure_hallucination_detection=measure_hallucination_detection).items():
            table[name].append(score)

    print(f"Processed {num_rows} rows.")
    print(f"Number of rows with JSON parse errors: {num_parse_errors}.")
    print(f"Number of rows where LLM failed to adhere to response schema: {num_gen_errors}.")
    print(f"Total number of erroneous rows: {num_parse_errors + num_gen_errors}.")
    print(f"Error rate: {(num_parse_errors + num_gen_errors) / num_rows}")

    return table

In [5]:
def bold_extreme_values(s, by_model=True):
    # Bold max for mean

    if 'mean' not in s.name and 'std' not in s.name:
        return ['' for v in s]


    if not by_model:
        if "mean" in s.name:
            is_max = s == s.max()
            return ['font-weight: bold' if v else '' for v in is_max]
        if "std" in s.name:
            is_min = s == s.min()
            return ['font-weight: bold' if v else '' for v in is_min]
    
    font_array = []

    model_level = s.index.names.index('model')
    models = s.index.get_level_values(model_level)
    models = pd.Series(list(models)).unique()

    idx = pd.IndexSlice
    
    for model in models:   
        values_by_model = s.loc[idx[model]]
        
        if "mean" in s.name:
            is_max = values_by_model == values_by_model.max()
            font_array += ['font-weight: bold' if v else '' for v in is_max]
        if "std" in s.name:
            is_min = values_by_model == values_by_model.min()
            font_array += ['font-weight: bold' if v else '' for v in is_min]

    return font_array

In [6]:
# Collect raw data
basepath = "./outputs/v5"
finished_jobs = collect_jobs(basepath)
jobs_list = [job for _, job_list in finished_jobs.items() for job in job_list] # Flatten

#res1 = pd.DataFrame(score_table(jobs_list, cols, mark_gen_errors_as_hallucinations=False))
#res1 = prettify_table(res1)

In [7]:
res2 = pd.DataFrame(score_table(jobs_list, cols, mark_gen_errors_as_hallucinations=True))
res2 = prettify_table(res2)

Processing 330 jobs.
Processed 176550 rows.
Number of rows with JSON parse errors: 1082.
Number of rows where LLM failed to adhere to response schema: 189.
Total number of erroneous rows: 1271.
Error rate: 0.007199093741149816


In [39]:
agg2 = aggregate_results(res2, ["model", "number_of_demonstrations", "type_of_demonstrations", "use_instructions"], cols[4:])

use_models=[
    "Llama-3.1-8B-Instruct",
    "Llama-3.3-70B-Instruct",
    "Mistral-7B-Instruct-v0.3",
    "Mistral-Small-3.2-24B-Instruct-2506",
    "Qwen3-8B",
    "Qwen3-32B"
]

idx = pd.IndexSlice

# Drop count
agg2 = agg2.loc[idx[use_models, :, :, :], (["theme_f1", "topic_f1", "concept_f1"], ["mean"])]
agg2 = format_table(agg2)

#agg2
# Highlight max values for each column in model group
agg2.style.apply(bold_extreme_values, axis=0)
#print(agg2.style.apply(bold_extreme_values, axis=0).to_latex())
#print(agg2.to_latex(escape=True, sparsify=True, float_format="%.3f"))

In [35]:
flattened = agg2.copy()

flattened.columns = [" ".join(list(cols)) for cols in flattened.columns]
flattened = flattened.reset_index()

flattened
#flattened.to_csv("./data/metrics.csv", sep=";")

,model,number of demonstrations,type of demonstrations,use instructions,total accuracy mean,total accuracy std,total accuracy count,theme accuracy mean,theme accuracy std,theme accuracy count,...,concept accuracy count,concept precision mean,concept precision std,concept precision count,concept recall mean,concept recall std,concept recall count,concept f1 mean,concept f1 std,concept f1 count
0,Llama-8B,0,none,yes,0.400374,0.004262,5,0.772336,0.002437,5,...,5,0.635041,0.006575,5,0.928727,0.005515,5,0.576297,0.014446,5
1,Llama-8B,1,negative,no,0.442243,0.065282,5,0.607103,0.063240,5,...,5,0.806672,0.156045,5,0.678545,0.211106,5,0.702853,0.101314,5
2,Llama-8B,1,negative,yes,0.462430,0.026977,5,0.667664,0.055816,5,...,5,0.600213,0.034609,5,0.971636,0.023199,5,0.452155,0.119492,5
3,Llama-8B,1,positive,no,0.413832,0.055363,5,0.557383,0.073902,5,...,5,0.603026,0.048046,5,0.624727,0.119348,5,0.568699,0.069979,5
4,Llama-8B,1,positive,yes,0.536822,0.060168,5,0.765607,0.036451,5,...,5,0.701456,0.062252,5,0.912000,0.042438,5,0.681521,0.097814,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,Qwen3-8B,6,mixed,yes,0.705794,0.027331,5,0.943178,0.007546,5,...,5,0.998561,0.003217,5,0.485818,0.052230,5,0.786138,0.016898,5
62,Qwen3-8B,6,negative,no,0.588411,0.042611,5,0.630280,0.056623,5,...,5,0.607563,0.036737,5,0.904727,0.059663,5,0.492257,0.148925,5
63,Qwen3-8B,6,negative,yes,0.656075,0.028562,5,0.930467,0.010186,5,...,5,0.996825,0.007099,5,0.421818,0.063141,5,0.765495,0.018999,5
64,Qwen3-8B,6,positive,no,0.637383,0.034844,5,0.811589,0.026743,5,...,5,0.996844,0.004383,5,0.469091,0.045708,5,0.780270,0.014770,5
